# TRUSTDESC · Reproducible Research Pipeline
**Independent implementation · automatic Colab workflow · evidence-first handover**

TRUSTDESC replaces potentially poisoned tool descriptions with descriptions inferred from source and checked against behavior. This notebook acquires its verified project modules automatically, detects hardware, runs SliceMin → DescGen → DynVer, compares original/lite/full descriptions using a fixed task agent, and packages the evidence.

**Run:** Runtime → Change runtime type → Python 3 → **T4 GPU** → Save. Then Runtime → **Run all**. No uploads, API keys, cell edits, terminal commands or review spreadsheets are required. Public package/model downloads need internet access.

Reference: Hengkai Ye, Zhechang Zhang, Jinyuan Jia and Hong Hu, [*TRUSTDESC*](https://arxiv.org/abs/2604.07536v1), §§4–5.

**Scientific scope:** The original 52-tool/208-task artifact has not been authenticated or recovered. The automatic experiment uses two explicitly authored pure tools/eight synthetic tasks, plus a pinned paper-linked public source for static analysis. These are separate from the paper's original benchmark. The free local model is an alternative to the paper's models. Generated reports never turn missing observations into scores or declare Phase 2 complete without essential evidence.

<div style="padding:18px;border-radius:12px;background:#eef5f8;color:#17384b;font-size:16px">
<b>1 · Source + provenance</b> → <b>2 · SliceMin</b> (reachable code + unresolved calls)<br><br>
→ <b>3 · DescGen</b> (local LLM + source evidence) → <b>4 · DynVer</b> (tasks → execution → judge → retested corrections)<br><br>
→ <b>5 · Original / lite / full</b> (fixed task agent) → <b>6 · Reports + Phase 3 ZIP</b>
</div>

Lite is the refined full description with its functionality list removed. External code is parsed as text; external MCP execution requires a validated isolated container. The authored fixture allowlist is not an untrusted-code sandbox.

## 1 · Verify and acquire the project
A fixed Git commit and per-file SHA-256 manifest bind the modules used by this notebook. A cached copy is rechecked; corrupt or partial downloads are replaced. A bootstrap failure stops before importing unchecked code and gives the exact diagnostic.

In [ ]:
import hashlib, io, json, os, platform, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path, PurePosixPath
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, HTML, Markdown, FileLink, SVG
if sys.version_info < (3, 11):
    raise RuntimeError("Python >=3.11 is required. Select a current Python 3 Colab runtime.")
VALIDATION = globals().get("_TRUSTDESC_VALIDATION", {})
SOURCE_REVISION = 'ac2a36ca274757aacaeb1801fab4128918a99d70'
SOURCE_HASHES = {'LICENSE': '153de023e3ae1a344dbd2ea57c003f3e5f1de56a04ddcdd62c9a8f8718efcb56', 'NOTICE.md': '55f3c1ac343f02369b7820fe92f8ab5067542a0b2ca1b2a5e79c664d2bb5f536', 'configs/colab_experiment.json': 'd62e9db19bcf232157eeb58634a649aa7a7c6d0ba5b95cb65aa4308a66d93c6e', 'configs/colab_models.json': '67ee505ec61e8c990afa5ad3b31f591703843cc83000467d9ae9d9f7c2c7b4e9', 'configs/default.json': 'f3f1a79b3682a5ccfb91889b997c898c8f6436acfc807f7454264a721c66ba52', 'configs/experiments.json': '4f8ad204a8a06964589a07ccfa1cbffd6aad2b3ce42d8001a6e940c497fb99a4', 'configs/models.json': '0ce09cda2b1b7d44ea0e2563fd5cd7fc0b13ea9937d76642be8ab99082542eaf', 'configs/original_benchmark_contract.json': '315bf32365db842a9d15de2a60ed66cfd40eb319165952b4ff9bc95d3be38a46', 'configs/reproduction.json': '33b5f40b3fa3b283e26f4a1d3c9f111a0c8b3dff6e6de1005fb76a739e98bfe1', 'configs/reproduction_resources.json': 'e045aa32bae3fbd1279dd497c4244f633fe3af0bcefa16c84845b75707b0542c', 'configs/small_model.json': '82b0b74e57e53bd009b0be70db0a592f93b4482770e1cbe2874ae4ce5bcdc41d', 'data/manifests/acquired_sources.json': 'a0af02ac5c83b0e4184a80fc6320e1cd4353b3a5a80fda9a0f4e42af077fe4d1', 'data/manifests/paper_attacks.json': '9008117a10915235d7608426d2cee994c66c10a6059e54600d8fe3a8ddb9d204', 'data/manifests/paper_fingerprint.json': 'deb836fccab4f9d2ed27a21e394e69fcaefd1bb45255a911ec53406e44e817d5', 'data/manifests/paper_servers.json': 'da10ff31937b55b010abb3e32c64980e0142d245a053ceb54f8f6db4f6408b5b', 'docs/PAPER_ANALYSIS.md': 'd0c94428e053c1377429ccd997bfab8099afb6d96e0e30a846ee812a2f599d98', 'notebooks/colab/TRUSTDESC_FULL_REPRODUCTION.ipynb': '42519c373ac1affb298bf86c185c25dad64eb80a8bf643b0c8f645889b784ab9', 'notebooks/colab/phase1_handover.ipynb': '5078d468b0b2bb4e46c7b5e9d0fba4d19e72b77165a57f169ab4279fe7ceec72', 'notebooks/colab/requirements-cpu.txt': 'a184f1cd359ce251b7ffa5b9614b3e2eba3ae0aedbc04925f504bdccccef5c08', 'notebooks/colab/requirements-gpu.txt': 'b32a5eac09049efadde5e90dbe4e254ce2c3bc713c333d7cc2d4002ace3ea909', 'pyproject.toml': '763ffd6604e8c7917c3fbc02f8914aee89e19cf8c97676e8dec0e3d46269bb91', 'reports/published_results.csv': '38cf3479d4f4e6d8b7793bec413f81d9b0cd0e3ed9f64cc6c0af0d34c9494324', 'requirements-colab.txt': 'dd1a61867da5212b0578b2e66ecc41cb80f05fc0bf762bf626dfc2be7c405606', 'scripts/run_automatic_reproduction.py': '089f747ac56fc13c6ffa7d240b55b23687de223818395d5f9404e26393f39f06', 'src/trustdesc/__init__.py': '12d86d50a0571ce3ebd850078c61d98ac5c6d595d4f845908e9793d719a034a6', 'src/trustdesc/__main__.py': '935a1c1166b0c1ea35a82256345000bf2c73ded718d77773bc27a71ecce28f7d', 'src/trustdesc/cli.py': '37cb907db7389bdfc677af17d70ab9a903dd31046bfd4f1c4ae85ff463c45256', 'src/trustdesc/common/__init__.py': 'ba7bae2bfbda88212991e7c3f35716ed4709876571771a21563dbaad0d6f0009', 'src/trustdesc/common/io.py': 'f9d4a2b73dc2dd08c8c34c9f386bf90ce27c68ce27eb1ccf514ab319aa7ffadf', 'src/trustdesc/common/models.py': '8941f61941fdac08fa9e3e1284e38b19a234cf8bc7a32fc7a19a6e95bb946846', 'src/trustdesc/common/transformers_model.py': '60836358e7c53e80e3f697fa2c26de386fe340ce610a5b4b5d6eaaab4d75cee9', 'src/trustdesc/dataset/__init__.py': '369dd879f56e5ca7457e59a791ab21d67974868d8835b45e2e69d26c0b826966', 'src/trustdesc/dataset/acquire.py': 'c6aa11b974731556bf504d65ac34b327e5d3165a9b85e52e1ad6ae5a13cbee7b', 'src/trustdesc/dataset/prepare.py': '0cc36c0a6fd9646ed9ddb4cf0b6fcd4b7ff561fec34d4aa9a21fb5839ceebf0b', 'src/trustdesc/descgen/__init__.py': 'dd28014be8c22e087a0581ae20db1b12f459aa52431b8f1cde1b56aa0ad9c86d', 'src/trustdesc/descgen/generate.py': '81e453e7c466d8543f99ffdc98a55daa924eb491e5792b6c195427e7f47496a9', 'src/trustdesc/descgen/prompts.py': 'bfd9b82c1e803670cbf01f4b17aafb5927a3ecb135522522e520dd414750af19', 'src/trustdesc/descgen/sanitize.py': '6e73bb035b70b24e5e3bd167424932466ca992448516cb7271047c4a5dc39f53', 'src/trustdesc/dynver/__init__.py': '9988f6a69ef0949b6c2fd521d31b73c50d0a24a9840ab2ca1cbd1a66864cea0c', 'src/trustdesc/dynver/verify.py': '3c647e3ba585b1917c0a1af2515847486c988ae33c82ecff8edf30be486bb459', 'src/trustdesc/evaluation/__init__.py': '6d3183575a9cfce54ab63b0b05d8f4934b344d424bd385d69681152907c37cfb', 'src/trustdesc/evaluation/metrics.py': 'bcf97f6ce97cfc07cc3eca8b48ff11d6ea0eef95b50c8c0e60af86b64d525837', 'src/trustdesc/experiments/__init__.py': '7da7ee32c6909d71611fe766b63b53e134c963f8090c4e680db020264ace9f4c', 'src/trustdesc/experiments/data.py': '68fcf26e1f5c4494b379f693950e2b8df79d47b9434f80e3e639a181ab0fe19d', 'src/trustdesc/experiments/export.py': '5aaa76a15662df81d9f7f691bae35a2c856739d7fd12bea1471740de23c1752d', 'src/trustdesc/experiments/models.py': '7d06d9c8c800713b95a67b2801ccd0fcc62cfaea441f5ff36429eadf32021417', 'src/trustdesc/experiments/report.py': '3c307c05a06c1f92f15411d403da2379dd896293d5aee23f641c08638329cbcc', 'src/trustdesc/experiments/review.py': 'a35982f1c86357b57900d246eb5dc8a53c199fde5198bf7f410eb13c999aa561', 'src/trustdesc/experiments/runtime.py': '083809253128c4202db65c429ef6920bfed33c2f8b72792fcfb4f427f072b2a3', 'src/trustdesc/experiments/setup.py': '166d43f1e6bc01f35f13c23ab5709244bad0e6b6ce20ec25eb5e6ee345af2885', 'src/trustdesc/experiments/storage.py': '7cdde88f8f16ba9fd9299187084e361f2924fa0596e330f3ce9004611ba89ded', 'src/trustdesc/experiments/workflow.py': 'e8ac248f4af86e3236ec551e19f7e0c6664dd0fde055eaa9d90ba0253c7c2e8a', 'src/trustdesc/fixtures.py': '10402c0c1895dc0944f19c8d74cbcf99f86184a7de9eb49cde4dd16d77e4b335', 'src/trustdesc/pipeline.py': 'e08fd52e16b31f6450801d13b7fd14cb709d4896fdab6dd239f255ac01750208', 'src/trustdesc/reproduction/__init__.py': '7885af062fda01ff1923032d657139cc51c340d914223e8cf486f65be6d7889f', 'src/trustdesc/reproduction/evidence.py': 'c2cfa2c73ee336f8230911377e28d6ef4576545e6bfb3b57af7b1d9fde8792db', 'src/trustdesc/reproduction/model.py': '8dc9c98429a54250d777dfcb4bdbaf19e29a621824f37c65152fdab4c8f1f2b3', 'src/trustdesc/reproduction/report.py': '25834f91f8a0e55d6515de88ca87364e634b5b58b0eeb1a5bc12bbaf899236ef', 'src/trustdesc/reproduction/resources.py': '34bb4a05b7974ee71b87eed1d8ba5acf3e2fb2be65b79d63f30df2f1ca042c60', 'src/trustdesc/reproduction/session.py': '9b30e8db949e264e1558201356958bf4ccbdce862511f7fdad9b174b8c594728', 'src/trustdesc/reproduction/workflow.py': '76bf7b94b159831ae7085dff7dbc8aba19d25651fabd097a3e0e395759ccacee', 'src/trustdesc/resources/synthetic_server.py.txt': '0e168c22f203bab5472edc9d8efb716db4fdc64b0ab429e2e58b4c369ac5ce7d', 'src/trustdesc/sandbox/__init__.py': '14c551e9312a889668e374d6e48a50fe7269e0a2d5d086eb92ea7180670292e4', 'src/trustdesc/sandbox/fixture_tools.py': '1d9eef84be79d722b56bf4d14cfaba231d386a21c1d442475a27344ce498d725', 'src/trustdesc/sandbox/mcp_container.py': '5bc0a49694ac485c261ac61a1851c930aea8cd35b189d7c90dc1dc6448d97f85', 'src/trustdesc/sandbox/runner.py': '629aac4d964d4d21f8c0b17fbef73b79f8dd65b15c6cf3eea217bad473649f62', 'src/trustdesc/slicemin/__init__.py': '613cc2e63da63c44cef6fc5dff53a1ebdf5c0746a73efc34e4bedf047ba05cc6', 'src/trustdesc/slicemin/python.py': 'df368e9ca7ddd0ed0d722902cd5bd849694d8628594d1781d3817aa76c7c3777', 'src/trustdesc/slicemin/research.py': 'f5d9316baba771059198ee6ab35800b30ce768505b590cc3e22aeb4a6dc03d10', 'tests/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'tests/fixtures/server.py': '0e168c22f203bab5472edc9d8efb716db4fdc64b0ab429e2e58b4c369ac5ce7d', 'tests/fixtures/tasks.jsonl': '0e3c872390d8eb12fcaac961dd32222db1447de6b68580c2e04d5f31bac22081', 'tests/integration/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'tests/integration/test_cli.py': 'bf7d54d9924a5b137bf2b366aa772f98cc9921988a18b039aa2e16cc2cd47095', 'tests/integration/test_pipeline.py': 'f14a0562daa9695df65acb8b90c0daac0b87b771dbcce86183444aea1153a802', 'tests/unit/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'tests/unit/test_colab_workflow.py': 'fd24ee2126679bf64e434346052d5015683fbfb8c49e1d7474670818ec0f4964', 'tests/unit/test_data_evaluation.py': '5d374c8f822da8795fa4702f0edd540e809147ef21e87298cbc62871a82fbaa4', 'tests/unit/test_descgen.py': '6bcf1da0b79bfe16e0a9c9ab11bc567b6917b69d84cc0ec0fd458c8b448cf72a', 'tests/unit/test_dynver.py': 'bf9d8e0b65c1c210f046c79733d684d6ab52f5a0199ae4963aef60738bd96b73', 'tests/unit/test_reproduction.py': '199ed8d826c5a5bd98e3079d2e6c0a3196fdc537216261dee2bb983dc8643099', 'tests/unit/test_slicemin.py': '8f641e5afa6f8b74eb83dd57c4cdc686bbe4db7620c38df6631520c1532611fc'}
BASE = Path(VALIDATION.get("workspace", "/content/trustdesc_automatic" if Path("/content").exists() else "./trustdesc_automatic")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = Path(VALIDATION.get("project", BASE / ("project-" + SOURCE_REVISION))).resolve()
def verified_project():
    return PROJECT.is_dir() and all((PROJECT / name).is_file() and not (PROJECT/name).is_symlink() and hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() == sha for name, sha in SOURCE_HASHES.items())
if not VALIDATION.get("project") and not verified_project():
    url = "https://codeload.github.com/SamarthRamesha/trustdesc-reproduction/zip/" + SOURCE_REVISION
    try:
        request = urllib.request.Request(url, headers={"User-Agent":"trustdesc-colab-research"})
        with urllib.request.urlopen(request, timeout=60) as response:
            raw = response.read(25_000_001)
        if len(raw) > 25_000_000:
            raise ValueError("Project archive exceeds 25 MB download limit")
        with zipfile.ZipFile(io.BytesIO(raw)) as archive:
            members = archive.infolist()
            if len(members) > 10000 or sum(m.file_size for m in members) > 100_000_000:
                raise ValueError("Project archive exceeds expanded limits")
            if len({m.filename for m in members}) != len(members):
                raise ValueError("Duplicate project archive members")
            by_relative = {}
            for member in members:
                p = PurePosixPath(member.filename)
                if p.is_absolute() or ".." in p.parts or "\\" in member.filename or ((member.external_attr >> 16) & 0o170000) == 0o120000:
                    raise ValueError("Unsafe project archive member")
                if len(p.parts) > 1 and not member.is_dir():
                    by_relative[PurePosixPath(*p.parts[1:]).as_posix()] = member
            checked = {}
            for name, sha in SOURCE_HASHES.items():
                data = archive.read(by_relative[name])
                if hashlib.sha256(data).hexdigest() != sha:
                    raise ValueError("Project SHA-256 mismatch: " + name)
                checked[name] = data
            # No archive extraction API: only checked allowlisted files written.
            PROJECT.mkdir(parents=True, exist_ok=True)
            for name, data in checked.items():
                target = PROJECT / name
                if not target.resolve().is_relative_to(PROJECT) or target.is_symlink():
                    raise ValueError("Project output path is linked or escapes root")
                target.parent.mkdir(parents=True, exist_ok=True)
                temporary = target.with_name(target.name + ".download-part")
                temporary.write_bytes(data)
                temporary.replace(target)
        if not verified_project():
            raise ValueError("Post-write project verification failed")
    except Exception as exc:
        raise RuntimeError(f"Verified project acquisition failed at {url}: {type(exc).__name__}: {exc}. Completed run outputs remain in {BASE}. Rerun after network recovery.") from exc
elif VALIDATION.get("project"):
    print("Explicit local software-validation source override; not a Colab/GPU validation.")
if not VALIDATION.get("project"):
    (PROJECT / "PORTABLE_MANIFEST.json").write_text(json.dumps({"git_commit":SOURCE_REVISION,"notebook":"TRUSTDESC_REPRODUCTION.ipynb","source_integrity":"per-file SHA-256"}))
sys.path.insert(0, str(PROJECT / "src"))
print(f"Project ready · Python {platform.python_version()} · source {SOURCE_REVISION[:12]} · {len(SOURCE_HASHES)} verified files")

## 2 · Detect hardware and install pinned inference dependencies
T4 uses Qwen2.5-Coder-1.5B-Instruct (about 3.1 GB of weights). CPU uses the 0.5B variant (about 1 GB); substitutions are recorded. GPU VRAM, host RAM, disk and library versions are checked. Loading is blocked when estimates exceed available resources. CPU inference can be substantially slower. No paid API is used.

In [ ]:
INSTALL_ERRORS = []
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
startup_disk = shutil.disk_usage(BASE).free
print(f"Startup disk available: {startup_disk / 1024**3:.1f} GiB")
if startup_disk < 8 * 1024**3:
    INSTALL_ERRORS.append({"component":"disk_preflight","status":"unavailable","reason":"Less than 8 GiB free before dependency installation"})
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
def install_logged(arguments, name):
    log = BASE / (name + ".txt")
    with log.open("w") as stream:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *arguments], stdout=stream, stderr=subprocess.STDOUT)
    if result.returncode:
        INSTALL_ERRORS.append({"component":name,"status":"failed","reason":log.read_text()[-6000:]})
        print(f"{name} failed. Diagnostic: {log.read_text()[-1200:]}")
    else: print(name + " ready")
if not VALIDATION.get("skip_install", False) and not INSTALL_ERRORS:
    cuda_host = False
    try:
        probe = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.free", "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        cuda_host = probe.returncode == 0
        print(probe.stdout.strip() if cuda_host else "CUDA GPU unavailable; selecting CPU-compatible installation")
    except (OSError, subprocess.TimeoutExpired):
        print("CUDA GPU unavailable; selecting CPU-compatible installation")
    # Library imports happen only after installation. Existing Colab torch
    # is replaced only when the exact selected pin is missing.
    torch_family = {"torch":"2.9.0", "torchvision":"0.24.0", "torchaudio":"2.9.0"}
    if any((installed(name) or "").split("+")[0] != pin for name,pin in torch_family.items()):
        # Keep Colab's preinstalled vision/audio extensions ABI-compatible
        # even though this text-only model does not use their functionality.
        index = "https://download.pytorch.org/whl/cu126" if cuda_host else "https://download.pytorch.org/whl/cpu"
        install_logged(["torch==2.9.0", "torchvision==0.24.0", "torchaudio==2.9.0", "--index-url", index], "torch-install")
    wanted = {"transformers":"4.57.1", "huggingface-hub":"0.36.0", "safetensors":"0.6.2", "tokenizers":"0.22.1", "numpy":"2.2.6"}
    if any(installed(name) != pin for name,pin in wanted.items()):
        install_logged(["-r", str(PROJECT / "requirements-colab.txt")], "inference-install")
elif VALIDATION.get("skip_install", False):
    print("Dependency installation skipped only by explicit software-validation harness")
from trustdesc.reproduction.session import Session
SESSION = Session(PROJECT, BASE / "experiment", network=not VALIDATION.get("offline", False),
                  model=VALIDATION.get("model"), model_error="Dependency installation failed; inspect install logs" if INSTALL_ERRORS else VALIDATION.get("model_error"))
SESSION.failures.extend(INSTALL_ERRORS)
for log in BASE.glob("*-install.txt"):
    SESSION.store.text(Path("diagnostics") / log.name, log.read_text())
display(Markdown("**Hardware/model decision**"))
display(SESSION.store.read("hardware_model_selection.json"))
print("Run directory:", SESSION.run_root)

## 3 · Acquire resources and validate provenance
The notebook checks the suggested artifact endpoint and author group, validates pinned external resources by SHA-256, and parses public dataset containers. MCPTox, MCPSecBench and MCPCorpus are external resources, not the TRUSTDESC benchmark. Their availability, licenses and missing components are written to the acquisition report. Unknown/unlicensed external data is referenced rather than bundled. Public tool code is never imported on the host.

In [ ]:
ACQUISITION = SESSION.acquire()
counts = {}
for resource in ACQUISITION["resources"]:
    counts[resource["status"]] = counts.get(resource["status"], 0) + 1
display(counts)
print("Original benchmark recovered:", ACQUISITION["original_artifact_recovered"])
print("Evaluated dataset:", SESSION.bundle["validation"])

## 4 · Initialize the free local model
The tokenizer and safetensors weights are loaded once from an immutable Hugging Face commit, with remote code disabled. Asset hashes, model settings and every prompt/raw response are recorded. Invalid responses get bounded format retries. An unavailable model produces explicit missing observations; it is never replaced with scripted semantic answers. Successful stage checkpoints remain recoverable.

In [ ]:
MODEL_MANIFEST = SESSION.initialize_model()
display({key: value for key,value in MODEL_MANIFEST.items() if key != "runtime"})
if MODEL_MANIFEST["status"] == "unavailable":
    print("Inference unavailable. Static analysis and honest missing-observation reports will continue.")

## 5 · SliceMin — reachable implementation and evidence
Python source is analyzed without execution. Supported aliases, imports, nested functions, self/cls methods and unambiguous dispatcher callees are followed. Unresolved calls remain visible. Model-guided debloat proposals are saved; semantic changes lacking equivalence evidence are not silently accepted. TypeScript and general dynamic resolution remain limitations of this independent implementation.

In [ ]:
STATIC = SESSION.workflow.static()
display({tool: {"status":result["status"], "reachable_functions":len(result.get("slice",{}).get("functions",[])),
               "unresolved_calls":len(result.get("slice",{}).get("unresolved_calls",[])),
               "diagnostic":result.get("error", result.get("reason"))} for tool,result in STATIC.items()})

## 6 · DescGen — grounded descriptions
Comments/docstrings are removed, identifiers normalized and contextually filtered, and descriptions generated from the code. All prose is decomposed into claims. Source quotations must match actual implementation spans; input names, required flags and simple annotated types must match the entry signature. This is evidence validation, not a proof of semantic equivalence.

In [ ]:
DRAFTS = SESSION.workflow.descriptions()
display({model:{tool:{"status":result["status"],"diagnostic":result.get("error",result.get("reason")),
                     "claims":len(result.get("draft",{}).get("description",{}).get("claims",[]))} for tool,result in tools.items()} for model,tools in DRAFTS.items()})

## 7 · DynVer — execute, judge and refine
A fixed task agent executes synthesized verification objectives against the authored tool allowlist. Claims are supported, refuted or inconclusive. Corrected descriptions must pass another execution/judging round. The offline container MCP transport stays disabled without a reviewed local image and successful isolation probes; missing infrastructure is not evidence that a claim is false. Read `tool_verification.json` for plans, rounds, decisions and uncertainty.

In [ ]:
VERIFIED = SESSION.workflow.verification()
display({model:{tool:{"status":result["status"],"rounds":result.get("verification",{}).get("refinement_rounds"),
                     "usable_summary":result.get("verification",{}).get("usable_summary"),
                     "diagnostic":result.get("error",result.get("reason"))} for tool,result in tools.items()} for model,tools in VERIFIED.items()})
display(SESSION.isolation)

## 8 · Matched original / lite / full evaluation
The same eight authored tasks, tool set and task-agent model/revision/settings are reused across all conditions. A condition is unavailable if a required trusted description is missing. An exact trace oracle checks the requested authored objective and target call. **This automatic fixture metric is distinct from the paper's manually reviewed TSR.** Original paper TSR, adaptive attack success and tool-modification rates stay unavailable without their required observations.

In [ ]:
TASK_RESULTS = SESSION.workflow.benchmark()
from collections import Counter
display(dict(Counter(result["status"] for result in TASK_RESULTS)))
print("Fixed task agent:", SESSION.config["task_agent"]["model_id"], SESSION.config["task_agent"]["revision"])

## 9 · Software validation
Unit and integration tests check software behavior separately from scientific evaluation. Output consistency, matched task-agent identity, provenance, checkpoint hashes and the ZIP member hashes are also checked before export. Tests do not supply labels for the original paper benchmark.

In [ ]:
import unittest
if not VALIDATION.get("skip_tests", False):
    test_log = BASE / "software-tests.txt"
    with test_log.open("w") as stream:
        test_result = subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-t","."], cwd=PROJECT,
                                     env={**os.environ,"PYTHONPATH":str(PROJECT/"src")}, stdout=stream, stderr=subprocess.STDOUT)
    SESSION.store.text("software_tests.txt",test_log.read_text())
    SESSION.store.write("software_test_status.json",{"returncode":test_result.returncode,"scope":"software tests, not research efficacy"})
    print(test_log.read_text()[-1600:])
    if test_result.returncode:
        SESSION.failures.append({"component":"software_tests","status":"failed","reason":"See software_tests.txt"})
else:
    SESSION.store.write("software_test_status.json",{"status":"skipped_by_explicit_validation_harness"})

## 10 · Results, limitations and Phase 3 handover
The reports distinguish completed observations, failed stages and missing original material. Generation time/tokens/API charges come only from saved calls. Runtime overhead uses the three-condition common-success intersection. Fixture Wilson intervals are descriptive references for authored tests, not population estimates. The package includes the exact source/configuration, dataset references, source slices, prompts, raw model records, traces, checkpoints, comparisons, manifests, reports and integrity hashes. Model caches, secrets, the private PDF and personal files are excluded.

In [ ]:
EXPORT = SESSION.finish()
RESULTS = SESSION.results
display(SVG(filename=str(SESSION.run_root / "plots/fixture_objectives.svg")))
table = [r for r in RESULTS["conditions"] if r["tool"] == "ALL"]
display(table)
display(Markdown((SESSION.run_root / "PHASE2_COMPLETION_REPORT.md").read_text()))
print("Output validation:", SESSION.store.read("validation_report.json")["valid"])
print("ZIP verified:", EXPORT["sha256"])
print("Indexed output files:", EXPORT["files"])
display(Markdown("### Download handover and reports"))
# Small report links are self-contained downloads; the ZIP also uses
# Colab's native browser downloader below.
def result_link(path):
    path = Path(path).resolve()
    if Path("/content").exists() and path.is_relative_to(Path("/content")):
        from html import escape
        if path.stat().st_size <= 2_000_000:
            import base64
            encoded = base64.b64encode(path.read_bytes()).decode("ascii")
            display(HTML(f'<a href="data:application/octet-stream;base64,{encoded}" download="{escape(path.name)}">{escape(path.name)}</a>'))
        else:
            relative = path.relative_to("/content").as_posix()
            display(HTML(f'<a href="/files/{escape(relative)}" download target="_blank">{escape(path.name)}</a>'))
    else:
        display(FileLink(str(path)))
result_link(EXPORT["archive"])
result_link(EXPORT["checksum_file"])
if not VALIDATION and SESSION.runtime["runtime_type"] == "colab":
    try:
        from google.colab import files
        files.download(EXPORT["archive"])
    except Exception as exc:
        print("Automatic browser download was unavailable; use the ZIP link above:", str(exc))
from trustdesc.reproduction.report import REPORTS
for report in REPORTS + ["results.json","validation_report.json","OUTPUT_INDEX.json"]:
    result_link(SESSION.run_root / report)
print("Generated output groups (complete file list is linked as OUTPUT_INDEX.json):")
output_names = SESSION.store.read("OUTPUT_INDEX.json")
print("\n".join(sorted({name.split("/")[0] + ("/" if "/" in name else "") for name in output_names})))
display(Markdown("**Pipeline finished. Phase 2 scientific status: INCOMPLETE.** Read the completion report for missing original artifacts and any failed stages. Download the ZIP before the Colab VM is discarded. Rerun unchanged cells to reuse completed checkpoints and retry failed stages."))